# Module Sécurité — Préparation et harmonisation (version révisée)

Ce notebook applique un nettoyage documenté sur le sous-domaine **Incidents sécuritaires**.

**Changements par rapport à la version précédente :**
- `conflict_data_bfa.csv` **exclu** : ce jeu présente un biais potentiel et n'est plus intégré à l'analyse.
- Fichiers annuels mis à jour vers les versions **03 avril 2026** (précédemment 27 mars 2026).
- Intégration des **3 fichiers HRP** ACLED Burkina Faso : ciblage civil, violence politique, manifestations — avec granularité Admin1 (région) / Admin2 (province), periodicite mensuelle, 1997–2026.
- Ajout de la **série mensuelle** violence politique par pays-mois-année.


## Sections du notebook

1. Chargement et inventaire des fichiers retenus
2. Fonctions utilitaires et journal des actions
3. Fichiers HRP — incidents par région / province (mensuel, par type)
4. Incidents hebdomadaires agrégés (Africa_aggregated)
5. Séries annuelles nationales (03 avril 2026)
6. Série mensuelle violence politique
7. Fusion et construction des séries annuelles harmonisées
8. Harmonisation finale
9. Exports et récapitulatif


In [1]:
from pathlib import Path
import json
import re
import unicodedata
import pandas as pd

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)


In [2]:
ROOT = Path(__file__).resolve().parent.parent if "__file__" in dir() else Path.cwd()
# Fallback robuste : remonter depuis le répertoire scripts/ si besoin
if not (ROOT / "data_raw").exists():
    ROOT = Path.cwd().parent if (Path.cwd().parent / "data_raw").exists() else Path.cwd()
RAW_DIR = ROOT / "data_raw" / "securite_stabilite" / "incidents_securitaires"
OUT_DIR = ROOT / "data" / "Securite" / "Incidents_securitaires"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Fichiers exclus de l'analyse
FICHIERS_EXCLUS = {
    "conflict_data_bfa.csv": "biais potentiel identifié — exclu à partir de la version révisée (S3)",
}

# Inventaire des fichiers retenus
files_retenus = sorted([
    p for p in RAW_DIR.iterdir()
    if p.is_file()
    and not p.name.startswith(".~lock")
    and p.name not in FICHIERS_EXCLUS
])

inventory = pd.DataFrame({
    "fichier": [p.name for p in files_retenus],
    "extension": [p.suffix.lower() for p in files_retenus],
    "taille_ko": [round(p.stat().st_size / 1024, 2) for p in files_retenus],
})
print(f"Fichiers retenus : {len(files_retenus)}")
print(f"Fichiers exclus  : {list(FICHIERS_EXCLUS.keys())}")
display(inventory)


Fichiers retenus : 13
Fichiers exclus  : ['conflict_data_bfa.csv']


,fichier,extension,taille_ko
0,Africa_aggregated_data_up_to_week_of-2026-03-2...,.xlsx,11551.51
1,burkina-faso_hrp_civilian_targeting_events_and...,.xlsx,566.57
2,burkina-faso_hrp_demonstration_events_by_month...,.xlsx,533.13
3,burkina-faso_hrp_political_violence_events_and...,.xlsx,587.53
4,number_of_demonstration_events_by_country-year...,.xlsx,47.70
5,number_of_events_targeting_civilians_by_countr...,.xlsx,44.21
6,number_of_events_targeting_civilians_by_countr...,.xlsx,44.24
7,number_of_political_violence_events_by_country...,.xlsx,448.72
8,number_of_political_violence_events_by_country...,.xlsx,45.03
9,number_of_reported_civilian_fatalities_by_coun...,.xlsx,44.04


In [3]:
action_logs = []

def log_action(section: str, action: str, details: str) -> None:
    action_logs.append({"section": section, "action": action, "details": details})

def to_snake(text: str) -> str:
    text = str(text).strip()
    text = unicodedata.normalize("NFKD", text).encode("ascii", "ignore").decode("ascii")
    text = text.lower()
    text = re.sub(r"[^a-z0-9]+", "_", text)
    text = re.sub(r"_+", "_", text).strip("_")
    return text

def normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out.columns = [to_snake(c) for c in out.columns]
    return out

def normalize_text(value):
    if pd.isna(value):
        return value
    return re.sub(r"\s+", " ", str(value).strip())

GEO_CORRECTIONS = {
    "sahel": "Sahel",
    "oudalan": "Oudalan",
    "centre-nord": "Centre-Nord",
    "centre nord": "Centre-Nord",
    "boucle du mouhoun": "Boucle du Mouhoun",
    "hauts bassins": "Hauts-Bassins",
    "hauts-bassins": "Hauts-Bassins",
    "sud-ouest": "Sud-Ouest",
    "sud ouest": "Sud-Ouest",
    "centre-est": "Centre-Est",
    "centre est": "Centre-Est",
    "centre-ouest": "Centre-Ouest",
    "centre ouest": "Centre-Ouest",
    "plateau central": "Plateau-Central",
    "plateau-central": "Plateau-Central",
}

def clean_geo_name(value):
    if pd.isna(value):
        return value
    v = normalize_text(value)
    return GEO_CORRECTIONS.get(v.lower(), v)

def drop_fully_empty_rows(df: pd.DataFrame):
    before = len(df)
    out = df.dropna(how="all").copy()
    return out, before - len(out)

MONTH_MAP = {
    "January": 1, "February": 2, "March": 3, "April": 4,
    "May": 5, "June": 6, "July": 7, "August": 8,
    "September": 9, "October": 10, "November": 11, "December": 12,
    "janvier": 1, "février": 2, "mars": 3, "avril": 4,
    "mai": 5, "juin": 6, "juillet": 7, "août": 8,
    "septembre": 9, "octobre": 10, "novembre": 11, "décembre": 12,
}

def normaliser_dataframe_fr(df: pd.DataFrame) -> pd.DataFrame:
    """Renomme les colonnes en français pour les exports."""
    rename_map = {
        "country": "pays", "admin1": "region", "admin2": "province",
        "month": "mois", "month_num": "mois_num", "year": "annee",
        "events": "nb_evenements", "fatalities": "nb_deces",
        "event_count": "nb_evenements", "fatalities_total": "nb_deces",
        "fatalities_civilians": "nb_deces_civils",
        "type_evenement": "type_evenement",
        "week": "date_semaine", "admin1_pcode": "code_region",
        "admin2_pcode": "code_province",
    }
    cols = {c: rename_map[c] for c in df.columns if c in rename_map}
    return df.rename(columns=cols)


## 1 — Fichiers HRP (granularité région / province, mensuel)

Ces trois fichiers ACLED Burkina Faso (feuille `Data`) couvrent :
- **Ciblage de civils** (`hrp_civilian_targeting`) : événements + décès par région/province/mois/année
- **Violence politique** (`hrp_political_violence`) : événements + décès par région/province/mois/année
- **Manifestations** (`hrp_demonstration`) : événements par région/province/mois/année

Ils constituent la source géolocalisée la plus fine disponible après exclusion de `conflict_data_bfa.csv`.


In [4]:
HRP_FILES = {
    "civilian_targeting": RAW_DIR / "burkina-faso_hrp_civilian_targeting_events_and_fatalities_by_month-year_as-of-18apr2025.xlsx",
    "political_violence": RAW_DIR / "burkina-faso_hrp_political_violence_events_and_fatalities_by_month-year_as-of-08apr2026.xlsx",
    "demonstrations":     RAW_DIR / "burkina-faso_hrp_demonstration_events_by_month-year_as-of-08apr2026.xlsx",
}

hrp_frames = []
for type_evt, path in HRP_FILES.items():
    raw = pd.read_excel(path, sheet_name="Data")
    print(f"Aperçu brut {type_evt} ({len(raw)} lignes) :")
    display(raw.head(3))

    df = normalize_columns(raw.copy())
    df, removed = drop_fully_empty_rows(df)
    log_action(f"hrp_{type_evt}", "supprimer_lignes_vides", f"{removed} ligne(s) retirée(s)")

    # Filtrer Burkina Faso (tous les fichiers HRP sont déjà BFA-only mais on sécurise)
    if "country" in df.columns:
        df = df[df["country"].str.contains("burkina", case=False, na=False)].copy()

    df["country"] = "Burkina Faso"
    df["admin1"] = df["admin1"].apply(clean_geo_name)
    df["admin2"] = df["admin2"].apply(clean_geo_name) if "admin2" in df.columns else None
    df["year"] = pd.to_numeric(df["year"], errors="coerce")
    df["month_num"] = df["month"].map(MONTH_MAP)
    df["type_evenement"] = type_evt
    df["events"] = pd.to_numeric(df["events"], errors="coerce").fillna(0)
    if "fatalities" in df.columns:
        df["fatalities"] = pd.to_numeric(df["fatalities"], errors="coerce").fillna(0)
    else:
        df["fatalities"] = 0

    log_action(f"hrp_{type_evt}", "lignes_retenues", f"{len(df)} lignes Burkina Faso")

    cols_keep = ["country", "admin1", "admin2", "year", "month", "month_num", "type_evenement", "events", "fatalities"]
    if "admin2_pcode" in df.columns:
        cols_keep.append("admin2_pcode")
    if "admin1_pcode" in df.columns:
        cols_keep.append("admin1_pcode")
    hrp_frames.append(df[[c for c in cols_keep if c in df.columns]])

hrp_clean = pd.concat(hrp_frames, ignore_index=True)
hrp_clean = hrp_clean.sort_values(["type_evenement", "year", "month_num", "admin1"], na_position="last").reset_index(drop=True)
print(f"\nHRP fusionné : {len(hrp_clean)} lignes — types : {hrp_clean['type_evenement'].unique()}")
print(f"Années : {sorted(hrp_clean['year'].dropna().astype(int).unique())}")
print(f"Régions : {sorted(hrp_clean['admin1'].dropna().unique())}")
hrp_clean.head(5)


Aperçu brut civilian_targeting (15300 lignes) :


,Country,Admin1,Admin2,ISO3,Admin2 Pcode,Admin1 Pcode,Month,Year,Events,Fatalities
0,Burkina Faso,Centre-Est,Kourittenga,BFA,BF4803,BF48,January,1997,0,0
1,Burkina Faso,Cascades,Comoe,BFA,BF4701,BF47,January,1997,0,0
2,Burkina Faso,Sud-Ouest,Noumbiel,BFA,BF5703,BF57,January,1997,0,0


Aperçu brut political_violence (15840 lignes) :


,Country,Admin1,Admin2,ISO3,Admin2 Pcode,Admin1 Pcode,Month,Year,Events,Fatalities
0,Burkina Faso,Hauts-Bassins,Tuy,BFA,BF5303,BF53,January,1997,0,0
1,Burkina Faso,Centre-Nord,Namentenga,BFA,BF4902,BF49,January,1997,0,0
2,Burkina Faso,Centre-Nord,Bam,BFA,BF4901,BF49,January,1997,0,0


Aperçu brut demonstrations (15840 lignes) :


,Country,Admin1,Admin2,ISO3,Admin2 Pcode,Admin1 Pcode,Month,Year,Events
0,Burkina Faso,Hauts-Bassins,Tuy,BFA,BF5303,BF53,January,1997,0
1,Burkina Faso,Centre-Nord,Namentenga,BFA,BF4902,BF49,January,1997,0
2,Burkina Faso,Centre-Nord,Bam,BFA,BF4901,BF49,January,1997,0



HRP fusionné : 46980 lignes — types : <StringArray>
['civilian_targeting', 'demonstrations', 'political_violence']
Length: 3, dtype: str
Années : [np.int64(1997), np.int64(1998), np.int64(1999), np.int64(2000), np.int64(2001), np.int64(2002), np.int64(2003), np.int64(2004), np.int64(2005), np.int64(2006), np.int64(2007), np.int64(2008), np.int64(2009), np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025), np.int64(2026)]
Régions : ['Boucle du Mouhoun', 'Cascades', 'Centre', 'Centre-Est', 'Centre-Nord', 'Centre-Ouest', 'Centre-Sud', 'Est', 'Hauts-Bassins', 'Nord', 'Plateau-Central', 'Sahel', 'Sud-Ouest']


,country,admin1,admin2,year,month,month_num,type_evenement,events,fatalities,admin2_pcode,admin1_pcode
0,Burkina Faso,Boucle du Mouhoun,Mouhoun,1997,January,1,civilian_targeting,0,0,BF4604,BF46
1,Burkina Faso,Boucle du Mouhoun,Sourou,1997,January,1,civilian_targeting,0,0,BF4606,BF46
2,Burkina Faso,Boucle du Mouhoun,Nayala,1997,January,1,civilian_targeting,0,0,BF4605,BF46
3,Burkina Faso,Boucle du Mouhoun,Kossi,1997,January,1,civilian_targeting,0,0,BF4603,BF46
4,Burkina Faso,Boucle du Mouhoun,Bale,1997,January,1,civilian_targeting,0,0,BF4601,BF46


## 2 — Incidents hebdomadaires (Africa_aggregated)

Fichier ACLED Afrique entière, filtré sur le Burkina Faso.


In [5]:
weekly_raw = pd.read_excel(RAW_DIR / "Africa_aggregated_data_up_to_week_of-2026-03-21.xlsx")
print(f"Aperçu brut Africa_aggregated ({len(weekly_raw)} lignes) :")
display(weekly_raw.head(3))

weekly = normalize_columns(weekly_raw.copy())
weekly, removed_w = drop_fully_empty_rows(weekly)
log_action("incidents_hebdo", "supprimer_lignes_vides", f"{removed_w} ligne(s) retirée(s)")

weekly = weekly[weekly["country"].astype(str).str.contains("burkina", case=False, na=False)].copy()
weekly["country"] = "Burkina Faso"
weekly["admin1"] = weekly["admin1"].apply(clean_geo_name)
weekly["week"] = pd.to_datetime(weekly["week"], errors="coerce")
weekly["year"] = weekly["week"].dt.year
weekly["events"] = pd.to_numeric(weekly["events"], errors="coerce").fillna(0)
weekly["fatalities"] = pd.to_numeric(weekly["fatalities"], errors="coerce").fillna(0)
log_action("incidents_hebdo", "lignes_retenues", f"{len(weekly)} lignes Burkina Faso")

COLS_HEBDO = ["country", "admin1", "week", "year", "event_type", "disorder_type", "events", "fatalities"]
weekly_clean = weekly[[c for c in COLS_HEBDO if c in weekly.columns]].copy()
print(f"\nHebdo nettoyé : {len(weekly_clean)} lignes — années : {sorted(weekly_clean['year'].dropna().astype(int).unique())}")
weekly_clean.head(3)


Aperçu brut Africa_aggregated (268511 lignes) :


,WEEK,REGION,COUNTRY,ADMIN1,EVENT_TYPE,SUB_EVENT_TYPE,EVENTS,FATALITIES,POPULATION_EXPOSURE,DISORDER_TYPE,ID,CENTROID_LATITUDE,CENTROID_LONGITUDE
0,2004-10-23,Northern Africa,Algeria,Adrar,Battles,Armed clash,1,2,NaN,Political violence,47.0,26.4839,-1.388
1,2005-04-23,Northern Africa,Algeria,Adrar,Battles,Armed clash,1,0,NaN,Political violence,47.0,26.4839,-1.388
2,2005-06-25,Northern Africa,Algeria,Adrar,Battles,Armed clash,1,14,NaN,Political violence,47.0,26.4839,-1.388



Hebdo nettoyé : 8232 lignes — années : [np.int64(1997), np.int64(1998), np.int64(1999), np.int64(2000), np.int64(2001), np.int64(2002), np.int64(2003), np.int64(2004), np.int64(2005), np.int64(2006), np.int64(2007), np.int64(2008), np.int64(2009), np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025), np.int64(2026)]


,country,admin1,week,year,event_type,disorder_type,events,fatalities
13032,Burkina Faso,Boucle du Mouhoun,2006-07-01,2006,Battles,Political violence,1,9
13033,Burkina Faso,Boucle du Mouhoun,2011-05-14,2011,Battles,Political violence,1,2
13034,Burkina Faso,Boucle du Mouhoun,2017-07-08,2017,Battles,Political violence,2,1


## 3 — Séries annuelles nationales (03 avril 2026)

Cinq fichiers agrégés au niveau pays/année, mis à jour vers les versions du 03 avril 2026 :
- Événements ciblant des civils
- Victimes civiles
- Victimes totales
- Événements de violence politique
- Manifestations / protestations


In [6]:
ANNUAL_FILES = {
    "events_civils":          RAW_DIR / "number_of_events_targeting_civilians_by_country-year_as-of-03Apr2026.xlsx",
    "deces_civils":           RAW_DIR / "number_of_reported_civilian_fatalities_by_country-year_as-of-03Apr2026.xlsx",
    "deces_totaux":           RAW_DIR / "number_of_reported_fatalities_by_country-year_as-of-03Apr2026.xlsx",
    "violence_politique":     RAW_DIR / "number_of_political_violence_events_by_country-year_as-of-03Apr2026.xlsx",
    "manifestations":         RAW_DIR / "number_of_demonstration_events_by_country-year_as-of-03Apr2026.xlsx",
}

def clean_annual(path, value_col: str, output_col: str, label: str) -> pd.DataFrame:
    raw = pd.read_excel(path)
    df = normalize_columns(raw.copy())
    df, removed = drop_fully_empty_rows(df)
    log_action(f"annuel_{label}", "supprimer_lignes_vides", f"{removed} ligne(s)")
    df = df[df["country"].astype(str).str.contains("burkina", case=False, na=False)].copy()
    df["country"] = "Burkina Faso"
    df["year"] = pd.to_numeric(df["year"], errors="coerce")
    df[value_col] = pd.to_numeric(df[value_col], errors="coerce").fillna(0)
    df = df.rename(columns={value_col: output_col})
    log_action(f"annuel_{label}", "lignes_retenues", f"{len(df)} lignes BFA, années {int(df['year'].min())}–{int(df['year'].max())}")
    return df[["country", "year", output_col]].sort_values("year").reset_index(drop=True)

events_civ_clean     = clean_annual(ANNUAL_FILES["events_civils"],      "events",      "nb_evenements_civils",    "events_civils")
deces_civ_clean      = clean_annual(ANNUAL_FILES["deces_civils"],        "fatalities",  "nb_deces_civils",         "deces_civils")
deces_total_clean    = clean_annual(ANNUAL_FILES["deces_totaux"],        "fatalities",  "nb_deces_totaux",         "deces_totaux")
violence_pol_clean   = clean_annual(ANNUAL_FILES["violence_politique"],  "events",      "nb_violence_politique",   "violence_politique")
manifest_clean       = clean_annual(ANNUAL_FILES["manifestations"],      "events",      "nb_manifestations",       "manifestations")

print("Aperçu séries annuelles BFA :")
display(events_civ_clean.tail(5))


Aperçu séries annuelles BFA :


,country,year,nb_evenements_civils
25,Burkina Faso,2022,709
26,Burkina Faso,2023,699
27,Burkina Faso,2024,436
28,Burkina Faso,2025,197
29,Burkina Faso,2026,61


## 4 — Violence politique mensuelle par pays (03 avril 2026)

Série mensuelle (COUNTRY / MONTH / YEAR / EVENTS) couvrant tous les pays ACLED.


In [7]:
monthly_raw = pd.read_excel(RAW_DIR / "number_of_political_violence_events_by_country-month-year_as-of-03Apr2026.xlsx")
monthly = normalize_columns(monthly_raw.copy())
monthly, removed_m = drop_fully_empty_rows(monthly)
log_action("violence_politique_mensuelle", "supprimer_lignes_vides", f"{removed_m} ligne(s)")

monthly = monthly[monthly["country"].astype(str).str.contains("burkina", case=False, na=False)].copy()
monthly["country"] = "Burkina Faso"
monthly["year"] = pd.to_numeric(monthly["year"], errors="coerce")
monthly["month_num"] = monthly["month"].map(MONTH_MAP)
monthly["events"] = pd.to_numeric(monthly["events"], errors="coerce").fillna(0)
monthly_clean = monthly[["country", "year", "month", "month_num", "events"]].sort_values(["year", "month_num"]).reset_index(drop=True)
monthly_clean = monthly_clean.rename(columns={"events": "nb_violence_politique"})
log_action("violence_politique_mensuelle", "lignes_retenues", f"{len(monthly_clean)} lignes BFA")

print(f"Mensuel violence politique BFA : {len(monthly_clean)} lignes")
display(monthly_clean.tail(5))


Mensuel violence politique BFA : 352 lignes


,country,year,month,month_num,nb_violence_politique
347,Burkina Faso,2025,December,12,123
348,Burkina Faso,2026,January,1,107
349,Burkina Faso,2026,February,2,106
350,Burkina Faso,2026,March,3,133
351,Burkina Faso,2026,April,4,10


## 5 — Fusion et construction de la série annuelle harmonisée

Jointure des 5 tables annuelles sur `country` × `year`.


In [8]:
annual_bfa = events_civ_clean.copy()
for df_right in [deces_civ_clean, deces_total_clean, violence_pol_clean, manifest_clean]:
    annual_bfa = annual_bfa.merge(df_right, on=["country", "year"], how="outer")

annual_bfa = annual_bfa.sort_values("year").reset_index(drop=True)
for col in ["nb_evenements_civils", "nb_deces_civils", "nb_deces_totaux", "nb_violence_politique", "nb_manifestations"]:
    annual_bfa[col] = annual_bfa[col].fillna(0).astype(int)

log_action("fusion_annuelle", "fusion_5_tables", f"{len(annual_bfa)} lignes, années {int(annual_bfa['year'].min())}–{int(annual_bfa['year'].max())}")
print(f"Série annuelle harmonisée : {len(annual_bfa)} lignes")
display(annual_bfa.tail(10))


Série annuelle harmonisée : 30 lignes


,country,year,nb_evenements_civils,nb_deces_civils,nb_deces_totaux,nb_violence_politique,nb_manifestations
20,Burkina Faso,2017,51,67,117,97,86
21,Burkina Faso,2018,132,177,303,233,106
22,Burkina Faso,2019,367,1348,2220,634,149
23,Burkina Faso,2020,362,1076,2299,666,86
24,Burkina Faso,2021,694,762,2360,1339,183
25,Burkina Faso,2022,709,1418,4234,1652,165
26,Burkina Faso,2023,699,2398,8499,1716,159
27,Burkina Faso,2024,436,2353,7487,1318,151
28,Burkina Faso,2025,197,881,5274,1371,169
29,Burkina Faso,2026,61,126,1021,356,26


## 6 — Harmonisation finale

Le fichier harmonisé principal combine les incidents hebdomadaires (Africa_aggregated) et les incidents HRP  
géolocalisés (région / province). `conflict_data_bfa.csv` est exclu de cette consolidation.


In [9]:
# Préparation d'un schéma commun HRP + hebdo pour la table harmonisée principale
hrp_for_harmo = hrp_clean.copy()
hrp_for_harmo = hrp_for_harmo.rename(columns={
    "events": "event_count", "fatalities": "fatalities_total", "admin1": "region", "admin2": "province"
})
hrp_for_harmo["source"] = "hrp_acled"
hrp_for_harmo["date_semaine"] = pd.NaT

weekly_for_harmo = weekly_clean.copy()
weekly_for_harmo = weekly_for_harmo.rename(columns={
    "admin1": "region", "events": "event_count", "fatalities": "fatalities_total"
})
weekly_for_harmo["source"] = "africa_aggregated_acled"
weekly_for_harmo["province"] = None
weekly_for_harmo["month"] = None
weekly_for_harmo["month_num"] = None
weekly_for_harmo["type_evenement"] = weekly_for_harmo.get("event_type", None)

COLS_HARMO = ["country", "region", "province", "year", "month", "month_num", "date_semaine",
              "type_evenement", "event_count", "fatalities_total", "source"]

def align_cols(df, cols):
    for c in cols:
        if c not in df.columns:
            df[c] = None
    return df[cols]

harmonized = pd.concat([
    align_cols(hrp_for_harmo, COLS_HARMO),
    align_cols(weekly_for_harmo, COLS_HARMO),
], ignore_index=True)

harmonized["year"] = pd.to_numeric(harmonized["year"], errors="coerce")
harmonized["event_count"] = pd.to_numeric(harmonized["event_count"], errors="coerce").fillna(0)
harmonized["fatalities_total"] = pd.to_numeric(harmonized["fatalities_total"], errors="coerce").fillna(0)
harmonized["country"] = harmonized["country"].fillna("Burkina Faso")
harmonized = harmonized.sort_values(["year", "month_num", "region"], na_position="last").reset_index(drop=True)

log_action("harmonisation_finale", "fusion_hrp_hebdo", f"{len(harmonized)} lignes — conflict_data_bfa.csv exclu")
print(f"Table harmonisée finale : {len(harmonized)} lignes")
print(f"Sources : {harmonized['source'].value_counts().to_dict()}")
harmonized.head(5)


Table harmonisée finale : 55212 lignes
Sources : {'hrp_acled': 46980, 'africa_aggregated_acled': 8232}


,country,region,province,year,month,month_num,date_semaine,type_evenement,event_count,fatalities_total,source
0,Burkina Faso,Boucle du Mouhoun,Mouhoun,1997,January,1,NaT,civilian_targeting,0,0,hrp_acled
1,Burkina Faso,Boucle du Mouhoun,Sourou,1997,January,1,NaT,civilian_targeting,0,0,hrp_acled
2,Burkina Faso,Boucle du Mouhoun,Nayala,1997,January,1,NaT,civilian_targeting,0,0,hrp_acled
3,Burkina Faso,Boucle du Mouhoun,Kossi,1997,January,1,NaT,civilian_targeting,0,0,hrp_acled
4,Burkina Faso,Boucle du Mouhoun,Bale,1997,January,1,NaT,civilian_targeting,0,0,hrp_acled


## 7 — Exports et récapitulatif


In [10]:
# Exports individuels
normaliser_dataframe_fr(hrp_clean).to_csv(OUT_DIR / "incidents_hrp_region_province_mensuel_bfa_clean.csv", index=False)
normaliser_dataframe_fr(weekly_clean).to_csv(OUT_DIR / "incidents_hebdo_bfa_clean.csv", index=False)
normaliser_dataframe_fr(events_civ_clean).to_csv(OUT_DIR / "evenements_civils_annuels_bfa_clean.csv", index=False)
normaliser_dataframe_fr(deces_civ_clean).to_csv(OUT_DIR / "deces_civils_annuels_bfa_clean.csv", index=False)
normaliser_dataframe_fr(deces_total_clean).to_csv(OUT_DIR / "deces_totaux_annuels_bfa_clean.csv", index=False)
normaliser_dataframe_fr(violence_pol_clean).to_csv(OUT_DIR / "violence_politique_annuelle_bfa_clean.csv", index=False)
normaliser_dataframe_fr(manifest_clean).to_csv(OUT_DIR / "manifestations_annuelles_bfa_clean.csv", index=False)
normaliser_dataframe_fr(monthly_clean).to_csv(OUT_DIR / "violence_politique_mensuelle_bfa_clean.csv", index=False)
normaliser_dataframe_fr(annual_bfa).to_csv(OUT_DIR / "series_annuelles_bfa_harmonisees.csv", index=False)
normaliser_dataframe_fr(harmonized).to_csv(OUT_DIR / "incidents_securitaires_harmonises_bfa.csv", index=False)

# Suppression des fichiers obsolètes générés par l'ancienne version (basés sur conflict_data_bfa)
import os
obsolete = ["incidents_geolocalises_bfa_clean.csv"]
for f in obsolete:
    p = OUT_DIR / f
    if p.exists():
        p.unlink()
        print(f"Supprimé (obsolète) : {f}")

fichiers_generes = [
    "incidents_hrp_region_province_mensuel_bfa_clean.csv",
    "incidents_hebdo_bfa_clean.csv",
    "evenements_civils_annuels_bfa_clean.csv",
    "deces_civils_annuels_bfa_clean.csv",
    "deces_totaux_annuels_bfa_clean.csv",
    "violence_politique_annuelle_bfa_clean.csv",
    "manifestations_annuelles_bfa_clean.csv",
    "violence_politique_mensuelle_bfa_clean.csv",
    "series_annuelles_bfa_harmonisees.csv",
    "incidents_securitaires_harmonises_bfa.csv",
]

synthese = {
    "domaine": "Securite",
    "sous_domaine": "Incidents_securitaires",
    "version": "revisee_S3",
    "fichiers_exclus": FICHIERS_EXCLUS,
    "nb_lignes_hrp": len(hrp_clean),
    "nb_lignes_hebdo": len(weekly_clean),
    "nb_lignes_harmonise": len(harmonized),
    "nb_lignes_serie_annuelle": len(annual_bfa),
    "nb_lignes_violence_mensuelle": len(monthly_clean),
    "types_evenements_hrp": list(hrp_clean["type_evenement"].unique()),
    "annees_couvertes_hrp": [int(hrp_clean["year"].min()), int(hrp_clean["year"].max())],
    "annees_couvertes_annuel": [int(annual_bfa["year"].min()), int(annual_bfa["year"].max())],
    "regions_hrp": sorted(hrp_clean["admin1"].dropna().unique().tolist()),
    "fichiers_generes": fichiers_generes,
    "journal_actions": action_logs,
}

with open(OUT_DIR / "synthese_preparation_securite.json", "w", encoding="utf-8") as f:
    json.dump(synthese, f, ensure_ascii=False, indent=2)

print("\n=== Exports terminés ===")
for fname in fichiers_generes:
    p = OUT_DIR / fname
    if p.exists():
        nb = len(pd.read_csv(p))
        print(f"  {fname} — {nb} lignes")


Supprimé (obsolète) : incidents_geolocalises_bfa_clean.csv

=== Exports terminés ===
  incidents_hrp_region_province_mensuel_bfa_clean.csv — 46980 lignes
  incidents_hebdo_bfa_clean.csv — 8232 lignes
  evenements_civils_annuels_bfa_clean.csv — 30 lignes
  deces_civils_annuels_bfa_clean.csv — 30 lignes
  deces_totaux_annuels_bfa_clean.csv — 30 lignes
  violence_politique_annuelle_bfa_clean.csv — 30 lignes
  manifestations_annuelles_bfa_clean.csv — 30 lignes
  violence_politique_mensuelle_bfa_clean.csv — 352 lignes
  series_annuelles_bfa_harmonisees.csv — 30 lignes


  incidents_securitaires_harmonises_bfa.csv — 55212 lignes


## Récapitulatif

Ce notebook (version révisée S3) prépare et harmonise les données brutes du sous-domaine **Incidents sécuritaires** pour le Burkina Faso.

**Fichier exclu :** `conflict_data_bfa.csv` — biais potentiel identifié. Le fichier `incidents_geolocalises_bfa_clean.csv` précédemment produit à partir de cette source a été supprimé.

**Sources retenues :**
| Fichier | Granularité | Périodicité | Type |
|---|---|---|---|
| `burkina-faso_hrp_civilian_targeting_*` | Région / Province | Mensuel | Ciblage civils |
| `burkina-faso_hrp_political_violence_*` | Région / Province | Mensuel | Violence politique |
| `burkina-faso_hrp_demonstration_*` | Région / Province | Mensuel | Manifestations |
| `Africa_aggregated_data_*` | Région | Hebdomadaire | Tous types |
| `number_of_*_as-of-03Apr2026.xlsx` (5 fichiers) | National | Annuel | Séries thématiques |
| `number_of_political_violence_*_by_country-month-year_*` | National | Mensuel | Violence politique |

**Fichiers produits :** 10 exports CSV + synthèse JSON — tous dans `data/Securite/Incidents_securitaires/`.
